In [177]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pd.set_option('display.max_columns', None)


In [82]:
def calculate_metrics(y_true, y_pred, model_name="Model"):
    """Calculate comprehensive evaluation metrics"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    }

In [74]:
ml_data= {}
ml_data['X_train'] = pd.read_csv('../data/ml_X_train.csv')
ml_data['X_val'] = pd.read_csv('../data/ml_X_val.csv')
ml_data['X_test'] = pd.read_csv('../data/ml_X_test.csv')
ml_data['y_train'] = pd.read_csv('../data/ml_y_train.csv')
ml_data['y_val'] = pd.read_csv('../data/ml_y_val.csv')
ml_data['y_test'] = pd.read_csv('../data/ml_y_test.csv')

In [220]:
# xgb_final_model = joblib.load("../models/xgb_final_model.pkl")
xgb_baseline_model = joblib.load("../models/xgb_model_baseline_model.pkl")

In [78]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model(r"E:\Programs\COMPANY\Sashflow\project_future_projection\rnd\models\xgboost_advanced_model.json")

In [85]:
p1 = loaded_model.predict(ml_data['X_val'])
p2 = xgb_model.predict(ml_data['X_val'])

In [86]:
print(calculate_metrics(ml_data['y_val'], p1, 'test'))
print(calculate_metrics(ml_data['y_val'], p2, 'test'))


{'MAE': 1449.3096923828125, 'MSE': 4248928.0, 'RMSE': np.float64(2061.2927982215433), 'R2': 0.8378171920776367}
{'MAE': 1449.3096923828125, 'MSE': 4248928.0, 'RMSE': np.float64(2061.2927982215433), 'R2': 0.8378171920776367}


In [224]:
main_df = pd.read_csv('../data/refactored_df.csv')
weather_df = pd.read_csv('../data/weather_data.csv')
trends_df = pd.read_csv('../data/customer_trends.csv')

In [225]:
# Step 1: Data Preparation and Merging
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Loading datasets...")
print(f"Main dataset shape: {main_df.shape}")
print(f"Weather dataset shape: {weather_df.shape}")
print(f"Trends dataset shape: {trends_df.shape}")

# Convert date columns
main_df['Date'] = pd.to_datetime(main_df['Date'])
main_df = main_df[['Date', 'Branch', 'Segment', 'Rating', 'Tonnage', 'Qty', 'Year', 'Month', 'Week']]
weather_df['Date'] = pd.to_datetime(weather_df['Date'])
trends_df['Month'] = pd.to_datetime(trends_df['Month'])

print("\nDate ranges:")
print(f"Main data: {main_df['Date'].min()} to {main_df['Date'].max()}")
print(f"Weather data: {weather_df['Date'].min()} to {weather_df['Date'].max()}")
print(f"Trends data: {trends_df['Month'].min()} to {trends_df['Month'].max()}")


Loading datasets...
Main dataset shape: (147595, 9)
Weather dataset shape: (456, 11)
Trends dataset shape: (82, 2)

Date ranges:
Main data: 2019-04-03 00:00:00 to 2024-03-30 00:00:00
Weather data: 2019-04-01 00:00:00 to 2025-10-01 00:00:00
Trends data: 2019-01-01 00:00:00 to 2025-10-01 00:00:00


In [226]:
grouped_df = (
    main_df.groupby([main_df['Date'].dt.to_period('M').dt.to_timestamp().rename('MonthStart'), 'Branch'])
    .agg({'Qty': 'sum'})
    .reset_index()
)
grouped_df.columns = ['Date', 'Branch', 'Qty']

In [227]:
new_df = (
    main_df.groupby([main_df['Date'].dt.to_period('M').dt.to_timestamp().rename('MonthStart'), 'Branch'])
    .agg({'Qty': 'sum'})
    .reset_index()
)
new_df.columns = ['Date', 'Branch', 'Qty']
new_df = new_df[new_df['Date'] >= '2023-04-01']
new_df.loc[new_df['Date'] >= '2023-04-01', 'Date'] = (
    new_df.loc[new_df['Date'] >= '2023-04-01', 'Date'] + pd.DateOffset(years=1)
)
new_df['Qty']=np.nan

In [228]:
concat = pd.concat([grouped_df, new_df])

In [229]:
concat

,Date,Branch,Qty
0,2019-04-01,BLR,2667.0
1,2019-04-01,COK,1756.0
2,2019-04-01,MAA,13121.5
3,2019-04-01,SBD,3564.5
4,2019-04-01,SBD1,5260.0
...,...,...,...
290,2025-03-01,BLR,NaN
291,2025-03-01,COK,NaN
292,2025-03-01,MAA,NaN
293,2025-03-01,SBD,NaN


In [230]:
trends_df.rename(columns={'Month': 'Date'}, inplace=True)

In [231]:
ml_df = concat.merge(weather_df, on=['Date', 'Branch']).merge(trends_df, on=['Date'])

In [233]:
ml_df['Seasonality_Level'] = ml_df['Date'].dt.month_name().str[:3].map({
    'Mar': 3, 'Dec': 3, 'Feb': 3, 'Apr': 3,
    'May': 2, 'Jan': 2,
    'Jun': 1, 'Jul': 1, 'Aug': 1, 'Sep': 1, 'Oct': 1, 'Nov': 1
})

In [234]:
ml_df

,Date,Branch,Qty,Min Temp,Max Temp,Avg Temp,Min Humidity,Max Humidity,Avg Humidity,Min Wind Speed,Max Wind Speed,Avg Wind Speed,Interest,Seasonality_Level
0,2019-04-01,BLR,2667.0,20.0,36.0,27.991643,13.0,100.0,49.857939,0.0,46.4,10.715181,71,3
1,2019-04-01,MAA,13121.5,26.0,40.0,31.565694,33.0,94.0,70.098611,0.0,27.7,12.295556,71,3
2,2019-04-01,SBD,3564.5,21.0,41.0,30.852273,14.0,94.0,47.340909,0.0,29.5,11.066903,71,3
3,2019-05-01,BLR,2446.5,19.0,36.0,27.343284,18.0,100.0,68.128901,0.0,40.7,12.634783,66,2
4,2019-05-01,MAA,11896.5,27.8,42.0,33.247446,20.0,91.0,66.135753,0.0,42.5,14.206452,66,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,2025-02-01,SBD,NaN,15.0,35.0,25.690923,13.0,100.0,53.346726,0.0,22.3,10.552679,40,3
271,2025-03-01,BLR,NaN,16.0,35.0,25.794624,11.0,98.0,50.341398,0.0,33.5,11.462769,65,3
272,2025-03-01,COK,NaN,23.0,37.0,28.886557,27.0,100.0,76.334426,0.7,22.0,7.503115,65,3
273,2025-03-01,MAA,NaN,22.8,38.4,29.397984,24.0,95.0,73.627688,0.0,25.9,9.904839,65,3


In [235]:
# =============================================================================
# 2. COMPREHENSIVE FEATURE ENGINEERING (FIXED VERSION)
# =============================================================================

print("\n2. COMPREHENSIVE FEATURE ENGINEERING")
print("-" * 50)

# Import required modules
from sklearn.preprocessing import LabelEncoder

def drop_nan_columns(df, threshold=1.0, verbose=True):
    """
    Drop columns with a fraction of NaN values above the given threshold.
    
    Parameters:
        df (pd.DataFrame): Input DataFrame.
        threshold (float): Fraction (0–1). Columns with NaN ratio >= threshold are dropped.
                           Default 1.0 means drop columns that are entirely NaN.
        verbose (bool): Whether to print information about dropped columns.
        
    Returns:
        pd.DataFrame: DataFrame with selected columns dropped.
        list: List of dropped column names.
    """
    # Calculate fraction of missing values per column
    nan_ratio = df.isna().mean()
    
    # Select columns to drop
    drop_cols = nan_ratio[nan_ratio >= threshold].index.tolist()
    
    # Drop them
    df_cleaned = df.drop(columns=drop_cols)
    
    if verbose:
        print(f"Dropped {len(drop_cols)} column(s) with ≥{threshold*100:.0f}% NaN values:")
        if drop_cols:
            print(drop_cols)
        else:
            print("No columns dropped.")
    
    return df_cleaned

def create_ml_features(df):
    """
    Create comprehensive features for machine learning models with proper NaN handling
    """
    print("Creating comprehensive features...")
    
    # Start with the base dataframe
    ml_df = df.copy()
    ml_df = drop_nan_columns(ml_df, 0.5)
    
    # Check data availability for each branch
    branch_counts = ml_df.groupby('Branch').size()
    print(f"  Data points per branch: {dict(branch_counts)}")
    
    # 1. TIME-BASED FEATURES
    print("  Creating time-based features...")
    ml_df['year'] = ml_df['Date'].dt.year
    ml_df['month'] = ml_df['Date'].dt.month
    ml_df['day'] = ml_df['Date'].dt.day
    ml_df['dayofweek'] = ml_df['Date'].dt.dayofweek
    ml_df['dayofyear'] = ml_df['Date'].dt.dayofyear
    ml_df['week'] = ml_df['Date'].dt.isocalendar().week
    ml_df['quarter'] = ml_df['Date'].dt.quarter
    
    # Cyclical encoding for time features
    ml_df['month_sin'] = np.sin(2 * np.pi * ml_df['month'] / 12)
    ml_df['month_cos'] = np.cos(2 * np.pi * ml_df['month'] / 12)
    ml_df['dayofweek_sin'] = np.sin(2 * np.pi * ml_df['dayofweek'] / 7)
    ml_df['dayofweek_cos'] = np.cos(2 * np.pi * ml_df['dayofweek'] / 7)
    ml_df['quarter_sin'] = np.sin(2 * np.pi * ml_df['quarter'] / 4)
    ml_df['quarter_cos'] = np.cos(2 * np.pi * ml_df['quarter'] / 4)
    
    # 2. LAG FEATURES (ADAPTIVE BASED ON DATA AVAILABILITY)
    print("  Creating lag features...")
    # Sort by date and branch for proper lag calculation
    ml_df = ml_df.sort_values(['Branch', 'Date'])
    
    # Use smaller lag periods appropriate for monthly data
    lag_periods = [1, 2, 3, 6, 12]  # months instead of days
    for lag in lag_periods:
        ml_df[f'qty_lag_{lag}m'] = ml_df.groupby('Branch')['Qty'].shift(lag)
        # Only create rolling mean if we have enough data
        if lag <= 3:  # Only for short lags
            ml_df[f'qty_lag_{lag}m_mean'] = ml_df.groupby('Branch')['Qty'].shift(lag).rolling(window=min(3, lag), min_periods=1).mean()
    
    # 3. ROLLING STATISTICS (ADAPTIVE WINDOWS)
    print("  Creating rolling statistics...")
    # Use smaller windows appropriate for monthly data
    rolling_windows = [2, 3, 6, 12]  # months
    for window in rolling_windows:
        # Use min_periods to handle insufficient data gracefully
        ml_df[f'qty_rolling_mean_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).mean().reset_index(0, drop=True)
        ml_df[f'qty_rolling_std_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).std().reset_index(0, drop=True)
        ml_df[f'qty_rolling_max_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).max().reset_index(0, drop=True)
        ml_df[f'qty_rolling_min_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).min().reset_index(0, drop=True)
        ml_df[f'qty_rolling_sum_{window}m'] = ml_df.groupby('Branch')['Qty'].rolling(window=window, min_periods=1).sum().reset_index(0, drop=True)
    
    # 4. EXPANDING STATISTICS
    print("  Creating expanding statistics...")
    ml_df['qty_expanding_mean'] = ml_df.groupby('Branch')['Qty'].expanding().mean().reset_index(0, drop=True)
    ml_df['qty_expanding_std'] = ml_df.groupby('Branch')['Qty'].expanding().std().reset_index(0, drop=True)
    ml_df['qty_expanding_max'] = ml_df.groupby('Branch')['Qty'].expanding().max().reset_index(0, drop=True)
    ml_df['qty_expanding_min'] = ml_df.groupby('Branch')['Qty'].expanding().min().reset_index(0, drop=True)
    
    # 5. SEASONAL FEATURES
    print("  Creating seasonal features...")
    # Monthly seasonality
    monthly_avg = ml_df.groupby('month')['Qty'].mean()
    ml_df['monthly_seasonality'] = ml_df['month'].map(monthly_avg)
    
    # Quarterly seasonality
    quarterly_avg = ml_df.groupby('quarter')['Qty'].mean()
    ml_df['quarterly_seasonality'] = ml_df['quarter'].map(quarterly_avg)
    
    # Day of week seasonality
    dow_avg = ml_df.groupby('dayofweek')['Qty'].mean()
    ml_df['dow_seasonality'] = ml_df['dayofweek'].map(dow_avg)
    
    # 6. WEATHER LAG FEATURES (ADAPTIVE)
    print("  Creating weather lag features...")
    weather_lags = [1, 2, 3, 6]  # months
    for lag in weather_lags:
        ml_df[f'temp_lag_{lag}m'] = ml_df.groupby('Branch')['Avg Temp'].shift(lag)
        ml_df[f'humidity_lag_{lag}m'] = ml_df.groupby('Branch')['Avg Humidity'].shift(lag)
        ml_df[f'wind_lag_{lag}m'] = ml_df.groupby('Branch')['Avg Wind Speed'].shift(lag)
    
    # 7. TRENDS LAG FEATURES
    print("  Creating trends lag features...")
    trends_lags = [1, 2, 3, 6]
    for lag in trends_lags:
        ml_df[f'trends_lag_{lag}m'] = ml_df['Interest'].shift(lag)
    
    # 8. PRODUCT FEATURES
    print("  Creating product features...")
    # Branch encoding
    branch_encoder = LabelEncoder()
    ml_df['branch_encoded'] = branch_encoder.fit_transform(ml_df['Branch'])
    
    # 9. INTERACTION FEATURES
    print("  Creating interaction features...")
    ml_df['temp_humidity_interaction'] = ml_df['Avg Temp'] * ml_df['Avg Humidity']
    ml_df['temp_wind_interaction'] = ml_df['Avg Temp'] * ml_df['Avg Wind Speed']
    
    # 10. STATISTICAL FEATURES
    print("  Creating statistical features...")
    # Temperature statistics
    ml_df['temp_range'] = ml_df['Max Temp'] - ml_df['Min Temp']
    ml_df['humidity_range'] = ml_df['Max Humidity'] - ml_df['Min Humidity']
    ml_df['wind_range'] = ml_df['Max Wind Speed'] - ml_df['Min Wind Speed']
    
    # Temperature deviation from historical average
    temp_avg = ml_df.groupby('month')['Avg Temp'].transform('mean')
    ml_df['temp_deviation'] = ml_df['Avg Temp'] - temp_avg
    
    # Humidity deviation from historical average
    humidity_avg = ml_df.groupby('month')['Avg Humidity'].transform('mean')
    ml_df['humidity_deviation'] = ml_df['Avg Humidity'] - humidity_avg
    
    # 11. BUSINESS FEATURES
    print("  Creating business features...")
    # Days since last sale (convert to months for monthly data)
    ml_df['months_since_last_sale'] = ml_df.groupby('Branch')['Date'].diff().dt.days / 30.44  # Approximate days per month
    
    # Sales momentum (recent vs historical average) - handle division by zero
    ml_df['sales_momentum_3m'] = np.where(
        ml_df['qty_expanding_mean'] != 0,
        ml_df['qty_rolling_mean_3m'] / ml_df['qty_expanding_mean'],
        1.0  # Default to 1 if no historical data
    )
    ml_df['sales_momentum_6m'] = np.where(
        ml_df['qty_expanding_mean'] != 0,
        ml_df['qty_rolling_mean_6m'] / ml_df['qty_expanding_mean'],
        1.0
    )
    
    # Market share by branch
    branch_total = ml_df.groupby('Date')['Qty'].transform('sum')
    ml_df['branch_market_share'] = np.where(
        branch_total != 0,
        ml_df['Qty'] / branch_total,
        0.0
    )
    
    # 12. HANDLE REMAINING NaN VALUES
    print("  Handling remaining NaN values...")
    
    # Fill NaN values in lag features with 0 (no previous data)
    lag_cols = [col for col in ml_df.columns if 'lag' in col]
    ml_df[lag_cols] = ml_df[lag_cols].fillna(0)
    
    # Fill NaN values in rolling features with the current value or mean
    rolling_cols = [col for col in ml_df.columns if 'rolling' in col]
    for col in rolling_cols:
        if col.endswith('_mean') or col.endswith('_sum'):
            ml_df[col] = ml_df[col].fillna(ml_df['Qty'])
        else:
            ml_df[col] = ml_df[col].fillna(ml_df[col].mean())
    
    # Fill NaN values in expanding features
    expanding_cols = [col for col in ml_df.columns if 'expanding' in col]
    ml_df[expanding_cols] = ml_df[expanding_cols].fillna(ml_df['Qty'])
    
    # Fill NaN values in other features
    other_nan_cols = ml_df.columns[ml_df.isnull().any()].tolist()
    for col in other_nan_cols:
        if ml_df[col].dtype in ['int64', 'float64']:
            ml_df[col] = ml_df[col].fillna(ml_df[col].median())
        else:
            ml_df[col] = ml_df[col].fillna(ml_df[col].mode()[0] if not ml_df[col].mode().empty else 0)
    
    print(f"  Feature engineering completed!")
    print(f"  Total features created: {ml_df.shape[1]}")
    
    return ml_df

# Apply feature engineering
ml_df = create_ml_features(ml_df)

# Display feature categories
feature_categories = {
    'Time-based': [col for col in ml_df.columns if any(x in col for x in ['year', 'month', 'day', 'week', 'quarter', 'sin', 'cos'])],
    'Lag features': [col for col in ml_df.columns if 'lag' in col],
    'Rolling statistics': [col for col in ml_df.columns if 'rolling' in col],
    'Expanding statistics': [col for col in ml_df.columns if 'expanding' in col],
    'Weather features': [col for col in ml_df.columns if any(x in col for x in ['Temp', 'Humidity', 'Wind'])],
    'Trends features': [col for col in ml_df.columns if 'trends' in col or 'Interest' in col],
    'Product features': [col for col in ml_df.columns if any(x in col for x in ['branch'])],
    'Interaction features': [col for col in ml_df.columns if 'interaction' in col],
    'Statistical features': [col for col in ml_df.columns if any(x in col for x in ['range', 'deviation', 'momentum'])],
    'Business features': [col for col in ml_df.columns if any(x in col for x in ['market_share', 'days_since'])],
}

print(f"\nFeature Categories:")
for category, features in feature_categories.items():
    print(f"  {category}: {len(features)} features")

# Check for missing values
print(f"\nMissing Values Analysis:")
missing_values = ml_df.isnull().sum()
missing_percentage = (missing_values / len(ml_df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Count', ascending=False)

print(missing_summary[missing_summary['Missing Count'] > 0].head(10))



2. COMPREHENSIVE FEATURE ENGINEERING
--------------------------------------------------
Creating comprehensive features...
Dropped 0 column(s) with ≥50% NaN values:
No columns dropped.
  Data points per branch: {'BLR': np.int64(71), 'COK': np.int64(62), 'MAA': np.int64(71), 'SBD': np.int64(71)}
  Creating time-based features...
  Creating lag features...
  Creating rolling statistics...
  Creating expanding statistics...
  Creating seasonal features...
  Creating weather lag features...
  Creating trends lag features...
  Creating product features...
  Creating interaction features...
  Creating statistical features...
  Creating business features...
  Handling remaining NaN values...
  Feature engineering completed!
  Total features created: 90

Feature Categories:
  Time-based: 16 features
  Lag features: 24 features
  Rolling statistics: 20 features
  Expanding statistics: 4 features
  Weather features: 9 features
  Trends features: 5 features
  Product features: 2 features
  Inter

In [237]:
# Test the fixed feature engineering function
print("Testing the fixed feature engineering function...")
print("=" * 60)

# Apply the fixed feature engineering
ml_df_fixed = create_ml_features(ml_df)

# Display feature categories
feature_categories = {
    'Time-based': [col for col in ml_df_fixed.columns if any(x in col for x in ['year', 'month', 'day', 'week', 'quarter', 'sin', 'cos'])],
    'Lag features': [col for col in ml_df_fixed.columns if 'lag' in col],
    'Rolling statistics': [col for col in ml_df_fixed.columns if 'rolling' in col],
    'Expanding statistics': [col for col in ml_df_fixed.columns if 'expanding' in col],
    'Weather features': [col for col in ml_df_fixed.columns if any(x in col for x in ['Temp', 'Humidity', 'Wind'])],
    'Trends features': [col for col in ml_df_fixed.columns if 'trends' in col or 'Interest' in col],
    'Product features': [col for col in ml_df_fixed.columns if any(x in col for x in ['branch'])],
    'Interaction features': [col for col in ml_df_fixed.columns if 'interaction' in col],
    'Statistical features': [col for col in ml_df_fixed.columns if any(x in col for x in ['range', 'deviation', 'momentum'])],
    'Business features': [col for col in ml_df_fixed.columns if any(x in col for x in ['market_share', 'months_since'])],
}

print(f"\nFeature Categories:")
for category, features in feature_categories.items():
    print(f"  {category}: {len(features)} features")

# Check for missing values
print(f"\nMissing Values Analysis:")
missing_values = ml_df_fixed.isnull().sum()
missing_percentage = (missing_values / len(ml_df_fixed)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Count', ascending=False)

# Show only columns with missing values
missing_cols = missing_summary[missing_summary['Missing Count'] > 0]
if len(missing_cols) > 0:
    print("Columns with missing values:")
    print(missing_cols.head(10))
else:
    print("✅ No missing values found!")

print(f"\nData shape: {ml_df_fixed.shape}")
print(f"Total features: {ml_df_fixed.shape[1]}")
print(f"Total samples: {ml_df_fixed.shape[0]}")

# Show sample of the data
print(f"\nSample of processed data:")
print(ml_df_fixed.head())


Testing the fixed feature engineering function...
Creating comprehensive features...
Dropped 0 column(s) with ≥50% NaN values:
No columns dropped.
  Data points per branch: {'BLR': np.int64(71), 'COK': np.int64(62), 'MAA': np.int64(71), 'SBD': np.int64(71)}
  Creating time-based features...
  Creating lag features...
  Creating rolling statistics...
  Creating expanding statistics...
  Creating seasonal features...
  Creating weather lag features...
  Creating trends lag features...
  Creating product features...
  Creating interaction features...
  Creating statistical features...
  Creating business features...
  Handling remaining NaN values...
  Feature engineering completed!
  Total features created: 90

Feature Categories:
  Time-based: 16 features
  Lag features: 24 features
  Rolling statistics: 20 features
  Expanding statistics: 4 features
  Weather features: 9 features
  Trends features: 5 features
  Product features: 2 features
  Interaction features: 2 features
  Statistic

In [238]:
ml_df

,Date,Branch,Qty,Min Temp,Max Temp,Avg Temp,Min Humidity,Max Humidity,Avg Humidity,Min Wind Speed,Max Wind Speed,Avg Wind Speed,Interest,Seasonality_Level,year,month,day,dayofweek,dayofyear,week,quarter,month_sin,month_cos,dayofweek_sin,dayofweek_cos,quarter_sin,quarter_cos,qty_lag_1m,qty_lag_1m_mean,qty_lag_2m,qty_lag_2m_mean,qty_lag_3m,qty_lag_3m_mean,qty_lag_6m,qty_lag_12m,qty_rolling_mean_2m,qty_rolling_std_2m,qty_rolling_max_2m,qty_rolling_min_2m,qty_rolling_sum_2m,qty_rolling_mean_3m,qty_rolling_std_3m,qty_rolling_max_3m,qty_rolling_min_3m,qty_rolling_sum_3m,qty_rolling_mean_6m,qty_rolling_std_6m,qty_rolling_max_6m,qty_rolling_min_6m,qty_rolling_sum_6m,qty_rolling_mean_12m,qty_rolling_std_12m,qty_rolling_max_12m,qty_rolling_min_12m,qty_rolling_sum_12m,qty_expanding_mean,qty_expanding_std,qty_expanding_max,qty_expanding_min,monthly_seasonality,quarterly_seasonality,dow_seasonality,temp_lag_1m,humidity_lag_1m,wind_lag_1m,temp_lag_2m,humidity_lag_2m,wind_lag_2m,temp_lag_3m,humidity_lag_3m,wind_lag_3m,temp_lag_6m,humidity_lag_6m,wind_lag_6m,trends_lag_1m,trends_lag_2m,trends_lag_3m,trends_lag_6m,branch_encoded,temp_humidity_interaction,temp_wind_interaction,temp_range,humidity_range,wind_range,temp_deviation,humidity_deviation,months_since_last_sale,sales_momentum_3m,sales_momentum_6m,branch_market_share
0,2019-04-01,BLR,2667.0,20.0,36.0,27.991643,13.0,100.0,49.857939,0.0,46.4,10.715181,71,3,2019,4,1,0,91,14,2,8.660254e-01,-5.000000e-01,0.000000,1.000000,1.224647e-16,-1.000000e+00,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.0,2667.000000,1786.969817,2667.00000,2667.000000,2667.000000,2667.000000,2098.133360,2667.000000,2667.000000,2667.000000,2667.000000,2606.522192,2667.000000,2667.000000,2667.000000,2667.000000,2997.037829,2667.000000,2667.000000,2667.000000,2667.000000,1776.836502,2667.0,2667.0,8593.533333,4980.622642,4346.044118,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0,1395.605644,299.935528,16.0,87.0,46.4,-1.517627,-12.569413,1.018397,1.000000,1.00000,0.137808
3,2019-05-01,BLR,2446.5,19.0,36.0,27.343284,18.0,100.0,68.128901,0.0,40.7,12.634783,66,2,2019,5,1,2,121,18,2,5.000000e-01,-8.660254e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00,2667.0,2667.0,0.0,0.00,0.0,0.00,0.0,0.0,2556.750000,155.917045,2667.00000,2446.500000,5113.500000,2556.750000,155.917045,2667.000000,2446.500000,5113.500000,2556.750000,155.917045,2667.000000,2446.500000,5113.500000,2556.750000,155.917045,2667.000000,2446.500000,5113.500000,2556.750000,155.917045,2667.0,2446.5,4266.210526,4980.622642,5158.814286,27.991643,49.857939,10.715181,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,71.0,0.0,0.0,0.0,0,1862.867859,345.476444,17.0,82.0,40.7,-2.173943,-0.741631,0.985545,1.000000,1.00000,0.124100
6,2019-06-01,BLR,1103.0,20.0,33.0,25.694986,36.0,100.0,78.798050,1.8,40.7,19.437744,58,1,2019,6,1,5,152,22,2,1.224647e-16,-1.000000e+00,-0.974928,-0.222521,1.224647e-16,-1.000000e+00,2446.5,2446.5,2667.0,2667.00,0.0,0.00,0.0,0.0,1774.750000,949.997961,2446.50000,1103.000000,3549.500000,2072.166667,846.532978,2667.000000,1103.000000,6216.500000,2072.166667,846.532978,2667.000000,1103.000000,6216.500000,2072.166667,846.532978,2667.000000,1103.000000,6216.500000,2072.166667,846.532978,2667.0,1103.0,2842.736842,4980.622642,3792.693548,27.343284,68.128901,12.634783,27.991643,49.857939,10.715181,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,66.0,71.0,0.0,0.0,0,2024.714801,499.452554,13.0,64.0,38.9,-2.445067,3.961861,1.018397,1.000000,1.00000,0.103704
9,2019-07-01,BLR,1222.5,20.0,31.0,24.534600,38.0,100.0,80.464043,3.6,44.6,22.617910,38,1,2019,7,1,0,182,27,3,-5.000000e-01,-8.660254e-01,0.000000,1.000000,-1.000000e+00,-1.836970e-16,1103.0,1103.0,2446.5,2556.75,2667.0,2667.00,0.0,0.0,1162.750000,84.499260,1222.50000,1103.000000,2325.500000,1590.666667,743.577893,2446.500000,1103.000000,4772.000000,1859.750000,811.312979,2667.000000,1103.000000,7439.000000

In [191]:
# Test the fixed feature engineering function
print("Testing the fixed feature engineering function...")
print("=" * 60)

# Apply the fixed feature engineering
ml_df_fixed = create_ml_features(ml_df)

# Display feature categories
feature_categories = {
    'Time-based': [col for col in ml_df_fixed.columns if any(x in col for x in ['year', 'month', 'day', 'week', 'quarter', 'sin', 'cos'])],
    'Lag features': [col for col in ml_df_fixed.columns if 'lag' in col],
    'Rolling statistics': [col for col in ml_df_fixed.columns if 'rolling' in col],
    'Expanding statistics': [col for col in ml_df_fixed.columns if 'expanding' in col],
    'Weather features': [col for col in ml_df_fixed.columns if any(x in col for x in ['Temp', 'Humidity', 'Wind'])],
    'Trends features': [col for col in ml_df_fixed.columns if 'trends' in col or 'Interest' in col],
    'Product features': [col for col in ml_df_fixed.columns if any(x in col for x in ['branch'])],
    'Interaction features': [col for col in ml_df_fixed.columns if 'interaction' in col],
    'Statistical features': [col for col in ml_df_fixed.columns if any(x in col for x in ['range', 'deviation', 'momentum'])],
    'Business features': [col for col in ml_df_fixed.columns if any(x in col for x in ['market_share', 'months_since'])],
}

print(f"\nFeature Categories:")
for category, features in feature_categories.items():
    print(f"  {category}: {len(features)} features")

# Check for missing values
print(f"\nMissing Values Analysis:")
missing_values = ml_df_fixed.isnull().sum()
missing_percentage = (missing_values / len(ml_df_fixed)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Count', ascending=False)

# Show only columns with missing values
missing_cols = missing_summary[missing_summary['Missing Count'] > 0]
if len(missing_cols) > 0:
    print("Columns with missing values:")
    print(missing_cols.head(10))
else:
    print("✅ No missing values found!")

print(f"\nData shape: {ml_df_fixed.shape}")
print(f"Total features: {ml_df_fixed.shape[1]}")
print(f"Total samples: {ml_df_fixed.shape[0]}")

# Show sample of the data
print(f"\nSample of processed data:")
print(ml_df_fixed.head())


Testing the fixed feature engineering function...
Creating comprehensive features...
Dropped 0 column(s) with ≥50% NaN values:
No columns dropped.
  Data points per branch: {'BLR': np.int64(71), 'COK': np.int64(62), 'MAA': np.int64(71), 'SBD': np.int64(71)}
  Creating time-based features...
  Creating lag features...
  Creating rolling statistics...
  Creating expanding statistics...
  Creating seasonal features...
  Creating weather lag features...
  Creating trends lag features...
  Creating product features...
  Creating interaction features...
  Creating statistical features...
  Creating business features...
  Handling remaining NaN values...
  Feature engineering completed!
  Total features created: 90

Feature Categories:
  Time-based: 16 features
  Lag features: 24 features
  Rolling statistics: 20 features
  Expanding statistics: 4 features
  Weather features: 9 features
  Trends features: 5 features
  Product features: 2 features
  Interaction features: 2 features
  Statistic

In [242]:
# Prepare the data using the fixed features
print("\nPreparing data with fixed features...")
print("=" * 50)

def prepare_ml_data_fixed(df, target_col='Qty'):
    """
    Prepare data for machine learning models with improved NaN handling
    """
    print("Preparing data for ML models...")
    
    # Remove rows with missing target values
    df_clean = df.dropna(subset=[target_col]).copy()
    print(f"  After removing missing target values: {len(df_clean)} samples")
    
    # Check for any remaining missing values
    missing_before = df_clean.isnull().sum().sum()
    print(f"  Missing values before cleaning: {missing_before}")
    
    # Fill any remaining missing values with appropriate strategies
    print("  Filling remaining missing values...")
    
    # Fill numerical columns with median
    numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numerical_cols:
        if col != target_col and df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            print(f"    Filled {col} with median: {df_clean[col].median():.2f}")
    
    # Fill categorical columns with mode
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df_clean[col].isnull().any():
            mode_val = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown'
            df_clean[col] = df_clean[col].fillna(mode_val)
            print(f"    Filled {col} with mode: {mode_val}")
    
    # Define feature columns (exclude target and non-predictive columns)
    exclude_cols = [target_col, 'Date', 'Year', 'Month', 'Week', 'Branch', 'Segment', 'Rating']
    feature_cols = [col for col in df_clean.columns if col not in exclude_cols]
    
    print(f"  Total features: {len(feature_cols)}")
    print(f"  Total samples: {len(df_clean)}")
    
    # Check final missing values
    missing_after = df_clean.isnull().sum().sum()
    print(f"  Missing values after cleaning: {missing_after}")
    
    if missing_after > 0:
        print("  Warning: Still have missing values!")
        missing_cols = df_clean.columns[df_clean.isnull().any()].tolist()
        print(f"  Columns with missing values: {missing_cols}")
    
    # Sort by date
    df_clean_sorted = df_clean.sort_values('Date')
    df_clean_sorted =df_clean_sorted[df_clean_sorted['Date'] >= '2024-04-01']
    return df_clean_sorted[feature_cols]

# Prepare the data using the fixed features
ml_data_fixed = prepare_ml_data_fixed(ml_df_fixed)



Preparing data with fixed features...
Preparing data for ML models...
  After removing missing target values: 275 samples
  Missing values before cleaning: 0
  Filling remaining missing values...
  Total features: 87
  Total samples: 275
  Missing values after cleaning: 0


In [243]:

ml_data_fixed

,Min Temp,Max Temp,Avg Temp,Min Humidity,Max Humidity,Avg Humidity,Min Wind Speed,Max Wind Speed,Avg Wind Speed,Interest,Seasonality_Level,year,month,day,dayofweek,dayofyear,week,quarter,month_sin,month_cos,dayofweek_sin,dayofweek_cos,quarter_sin,quarter_cos,qty_lag_1m,qty_lag_1m_mean,qty_lag_2m,qty_lag_2m_mean,qty_lag_3m,qty_lag_3m_mean,qty_lag_6m,qty_lag_12m,qty_rolling_mean_2m,qty_rolling_std_2m,qty_rolling_max_2m,qty_rolling_min_2m,qty_rolling_sum_2m,qty_rolling_mean_3m,qty_rolling_std_3m,qty_rolling_max_3m,qty_rolling_min_3m,qty_rolling_sum_3m,qty_rolling_mean_6m,qty_rolling_std_6m,qty_rolling_max_6m,qty_rolling_min_6m,qty_rolling_sum_6m,qty_rolling_mean_12m,qty_rolling_std_12m,qty_rolling_max_12m,qty_rolling_min_12m,qty_rolling_sum_12m,qty_expanding_mean,qty_expanding_std,qty_expanding_max,qty_expanding_min,monthly_seasonality,quarterly_seasonality,dow_seasonality,temp_lag_1m,humidity_lag_1m,wind_lag_1m,temp_lag_2m,humidity_lag_2m,wind_lag_2m,temp_lag_3m,humidity_lag_3m,wind_lag_3m,temp_lag_6m,humidity_lag_6m,wind_lag_6m,trends_lag_1m,trends_lag_2m,trends_lag_3m,trends_lag_6m,branch_encoded,temp_humidity_interaction,temp_wind_interaction,temp_range,humidity_range,wind_range,temp_deviation,humidity_deviation,months_since_last_sale,sales_momentum_3m,sales_momentum_6m,branch_market_share
229,26.6,39.0,31.515278,31.0,92.0,72.612500,0.0,27.7,11.727083,92,3,2024,4,1,0,92,14,2,8.660254e-01,-5.000000e-01,0.000000,1.000000,1.224647e-16,-1.000000e+00,21649.0,21649.0,13451.0,11582.75,9714.5,11427.166667,5695.0,19518.5,12368.25,13124.962519,21649.0,3087.5,24736.5,12729.166667,9301.779565,21649.0,3087.5,38187.5,12078.166667,7186.639031,21649.0,3087.5,72469.0,9794.333333,5788.739804,21649.0,3087.5,117532.0,7987.691667,5073.079199,22719.0,1130.0,7434.368421,4631.123077,4106.321429,29.282661,72.563172,10.352957,27.790661,74.606322,9.481034,26.239785,77.649194,7.772581,29.112500,80.241935,8.637634,53.0,34.0,28.0,30.0,2,2288.403108,369.582289,12.4,61.0,27.7,2.006007,10.185148,1.018397,1.593598,1.512097,0.25
227,20.0,37.0,28.940000,11.0,94.0,42.493056,1.8,27.7,11.957361,92,3,2024,4,1,0,92,14,2,8.660254e-01,-5.000000e-01,0.000000,1.000000,1.224647e-16,-1.000000e+00,7137.0,7137.0,4024.0,4174.50,4325.0,3418.500000,2246.0,3119.0,5112.25,2863.428910,7137.0,3087.5,10224.5,4749.500000,2119.994163,7137.0,3087.5,14248.5,4084.000000,1617.523323,7137.0,2646.5,24504.0,3139.416667,1516.007553,7137.0,1309.5,37673.0,2257.841667,1263.103762,7137.0,375.0,7434.368421,4631.123077,4106.321429,26.820565,45.533602,11.629032,23.902299,55.979885,12.298420,21.848118,66.743280,11.909543,24.041129,68.670699,10.454032,53.0,34.0,28.0,30.0,0,1229.749028,346.046031,17.0,83.0,25.9,-0.569270,-19.934296,1.018397,2.103558,1.808807,0.25
228,25.0,38.0,30.990278,31.0,100.0,75.025000,1.8,30.0,7.883194,92,3,2024,4,1,0,92,14,2,8.660254e-01,-5.000000e-01,0.000000,1.000000,1.224647e-16,-1.000000e+00,10894.5,10894.5,5498.0,3988.25,2478.5,3429.666667,1323.5,3579.0,6991.00,5520.382641,10894.5,3087.5,13982.0,6493.333333,3997.540317,10894.5,3087.5,19480.0,4961.500000,3227.671932,10894.5,2373.5,29769.0,3595.750000,2692.647198,10894.5,1062.5,43149.0,2121.843137,1764.377470,10894.5,22.0,7434.368421,4631.123077,4106.321429,30.036828,72.215054,8.187769,28.926580,69.438218,6.952730,27.837500,74.525538,6.379973,27.596371,84.411290,6.522177,53.0,34.0,28.0,30.0,1,2325.045590,244.302386,13.0,69.0,28.2,1.481007,12.597648,1.018397,3.060233,2.338297,0.25
230,23.0,41.0,31.690833,14.0,83.0,42.745833,1.8,27.7,11.929306,92,3,2024,4,1,0,92,14,2,8.660254e-01,-5.000000e-01,0.000000,1.000000,1.224647e-16,-1.000000e+00,22408.0,22408.0,12797.5,9591.25,6385.0,11939.000000,6570.5,7147.5,12747.75,13661.656566,22408.0,3087.5,25495.5,12764.333333,9660.292702,22408.0,3087.5,38293.0,12351.666667,8776.567630,23411.0,3087.5,74110.0,8670.333333,7194.089665,23411.0,1841.5,104044.0,5997.475000,5090.909762,23411.0,1307.0,7434.368421,4631.123077,4106.321429,28.688844,50.551075,11.830914,25.637931,57.461207,12.686638,22.811

In [244]:
# loaded_model.predict(test)
result = xgb_baseline_model.predict(ml_data_fixed)

In [249]:
ml_df_fixed[ml_df_fixed['Date'] >= '2024-04-01']['Qty'] = result

In [252]:
result

array([5003.3804, 6317.7183, 5413.4175, 6762.5317, 3494.7158, 3568.3992,
       3935.6848, 3690.0425, 3129.4365, 2827.0742, 3202.3086, 3386.3994,
       3097.7437, 3393.9915, 2812.8206, 3189.311 , 3280.4482, 3463.3752,
       3261.4336, 2742.624 , 3524.2114, 3607.0657, 3122.389 , 3473.317 ,
       3044.916 , 3606.908 , 3542.4949, 3401.7659, 3241.619 , 3317.3525,
       3324.2659, 3148.357 , 3771.07  , 4623.427 , 4055.5774, 3959.466 ,
       3626.4443, 3890.1995, 3679.5957, 3419.396 , 3358.3296, 3491.9272,
       3415.0852, 3604.2134, 3637.041 , 3311.67  , 3702.7266, 3619.362 ],
      dtype=float32)

In [251]:
ml_df_fixed[ml_df_fixed['Date'] >= '2024-04-01']

,Date,Branch,Qty,Min Temp,Max Temp,Avg Temp,Min Humidity,Max Humidity,Avg Humidity,Min Wind Speed,Max Wind Speed,Avg Wind Speed,Interest,Seasonality_Level,year,month,day,dayofweek,dayofyear,week,quarter,month_sin,month_cos,dayofweek_sin,dayofweek_cos,quarter_sin,quarter_cos,qty_lag_1m,qty_lag_1m_mean,qty_lag_2m,qty_lag_2m_mean,qty_lag_3m,qty_lag_3m_mean,qty_lag_6m,qty_lag_12m,qty_rolling_mean_2m,qty_rolling_std_2m,qty_rolling_max_2m,qty_rolling_min_2m,qty_rolling_sum_2m,qty_rolling_mean_3m,qty_rolling_std_3m,qty_rolling_max_3m,qty_rolling_min_3m,qty_rolling_sum_3m,qty_rolling_mean_6m,qty_rolling_std_6m,qty_rolling_max_6m,qty_rolling_min_6m,qty_rolling_sum_6m,qty_rolling_mean_12m,qty_rolling_std_12m,qty_rolling_max_12m,qty_rolling_min_12m,qty_rolling_sum_12m,qty_expanding_mean,qty_expanding_std,qty_expanding_max,qty_expanding_min,monthly_seasonality,quarterly_seasonality,dow_seasonality,temp_lag_1m,humidity_lag_1m,wind_lag_1m,temp_lag_2m,humidity_lag_2m,wind_lag_2m,temp_lag_3m,humidity_lag_3m,wind_lag_3m,temp_lag_6m,humidity_lag_6m,wind_lag_6m,trends_lag_1m,trends_lag_2m,trends_lag_3m,trends_lag_6m,branch_encoded,temp_humidity_interaction,temp_wind_interaction,temp_range,humidity_range,wind_range,temp_deviation,humidity_deviation,months_since_last_sale,sales_momentum_3m,sales_momentum_6m,branch_market_share
227,2024-04-01,BLR,3087.5,20.0,37.0,28.940000,11.0,94.0,42.493056,1.8,27.7,11.957361,92,3,2024,4,1,0,92,14,2,8.660254e-01,-5.000000e-01,0.000000,1.000000,1.224647e-16,-1.000000e+00,7137.0,7137.0,4024.0,4174.50,4325.0,3418.500000,2246.0,3119.0,5112.25,2863.428910,7137.0,3087.5,10224.5,4749.500000,2119.994163,7137.0,3087.5,14248.5,4084.000000,1617.523323,7137.0,2646.5,24504.0,3139.416667,1516.007553,7137.0,1309.5,37673.0,2257.841667,1263.103762,7137.0,375.0,7434.368421,4631.123077,4106.321429,26.820565,45.533602,11.629032,23.902299,55.979885,12.298420,21.848118,66.743280,11.909543,24.041129,68.670699,10.454032,53.0,34.0,28.0,30.0,0,1229.749028,346.046031,17.0,83.0,25.9,-0.569270,-19.934296,1.018397,2.103558,1.808807,0.25
231,2024-05-01,BLR,3087.5,20.0,37.0,26.668011,18.0,100.0,67.383065,1.8,44.6,12.499731,100,2,2024,5,1,2,122,18,2,5.000000e-01,-8.660254e-01,0.974928,-0.222521,1.224647e-16,-1.000000e+00,3087.5,3087.5,7137.0,5580.50,4024.0,3877.666667,2646.5,2624.0,3087.50,0.000000,3087.5,3087.5,6175.0,4437.333333,2337.979915,7137.0,3087.5,13312.0,4157.500000,1547.649896,7137.0,3087.5,24945.0,3178.041667,1507.562941,7137.0,1309.5,38136.5,2271.442623,1257.030128,7137.0,375.0,4061.217391,4631.123077,4773.453488,28.940000,42.493056,11.957361,26.820565,45.533602,11.629032,23.902299,55.979885,12.298420,22.469444,79.108333,11.598194,92.0,53.0,34.0,26.0,0,1796.972289,333.342966,17.0,82.0,42.8,-2.849216,-1.487468,0.985545,1.953531,1.830335,0.25
235,2024-06-01,BLR,3087.5,20.0,33.0,24.359722,46.0,100.0,76.313889,3.6,37.1,17.525694,68,1,2024,6,1,5,153,22,2,1.224647e-16,-1.000000e+00,-0.974928,-0.222521,1.224647e-16,-1.000000e+00,3087.5,3087.5,3087.5,5112.25,7137.0,5162.000000,3284.0,1880.5,3087.50,0.000000,3087.5,3087.5,6175.0,3087.500000,0.000000,3087.5,3087.5,9262.5,4124.750000,1571.722837,7137.0,3087.5,24748.5,3278.625000,1452.376910,7137.0,1309.5,39343.5,2284.604839,1250.984485,7137.0,375.0,2885.304348,4631.123077,3595.895349,26.668011,67.383065,12.499731,28.940000,42.493056,11.957361,26.820565,45.533602,11.629032,21.308737,76.924731,12.026882,100.0,92.0,53.0,27.0,0,1858.985135,426.921048,13.0,54.0,33.5,-3.780331,1.477700,1.018397,1.351437,1.805454,0.25
239,2024-07-01,BLR,3087.5,20.0,31.0,23.513441,51.0,94.0,78.372312,7.6,41.0,22.128898,44,1,2024,7,1,0,183,27,3,-5.000000e-01,-8.660254e-01,0.000000,1.000000,-1.000000e+00,-1.836970e-16,3087.5,3087.5,3087.5,3087.50,3087.5,4749.500000,4325.0,1309.5,3087.50,0.000000,3087.5,3087.5,6175.0,3087.500000,0.000000,3087.5,3087.5,9262.5,3918.500000,1620.624355,7137.0,3087.5,23511.0,3426.791667,1317.677734,7137.0,2246.0,41121.5,2297.349206,1244.971183,7137.0,375.0,2344.173913,2819.217391,4106.321429,

In [254]:
fs_df = pd.read_csv('../data/Final Sales.csv')

In [262]:
fs_df['Date'] = pd.to_datetime(fs_df['Date'])

In [267]:
grouped_data = (
    fs_df.groupby([fs_df['Date'].dt.to_period('M').dt.to_timestamp().rename('MonthStart'), 'Branch'])
    .agg({'Sales Qty.': 'sum'})
    .reset_index()
)
grouped_data.columns = ['Date', 'Branch', 'Qty']

In [270]:
grouped_data[(grouped_data['Date'] >= '2024-04-01') & (grouped_data['Date'] <= '2025-03-01') & (grouped_data['Branch'] != 'VIJAYAWADA')]

,Date,Branch,Qty
138,2024-04-01,BANGALORE,6232.0
139,2024-04-01,CHENNAI,37511.0
140,2024-04-01,COCHIN,13241.0
141,2024-04-01,HYDERABAD,23754.0
143,2024-05-01,BANGALORE,4523.0
144,2024-05-01,CHENNAI,20702.0
145,2024-05-01,COCHIN,6120.0
146,2024-05-01,HYDERABAD,11406.0
148,2024-06-01,BANGALORE,2515.0
149,2024-06-01,CHENNAI,6720.0
